# Preprocessing & Search Tests

## Objectifs

Nous testons plusieurs stratégies de recherche sur le dataset de tickets SAV multilingue, en passant par:

1. **Preprocessing léger**: nettoyage texte, suppression boilerplate & PII
2. **BM25**: baseline lexical (exact/keyword matching)
3. **Categorical Search**: filtrage par métadonnées (type, queue, priority, tags)
4. **Semantic Search**: recherche vectorielle k-NN via OpenSearch
5. **Hybrid RRF**: fusion BM25 + sémantique via Reciprocal Rank Fusion
6. **Multimodal Ranking**: score sémantique + boost catégoriel
7. **Cross-Encoder Reranking**: reranking fin par cross-encoder
8. **Meta-Ranking**: ensemble de tous les rankings

**Dataset** : environ 4000 tickets SAV, 5 langues (EN, DE, FR, ES, PT), 17 colonnes  
**Stack** : OpenSearch 2.13, `paraphrase-multilingual-MiniLM-L12-v2` (dim=384)

In [ ]:
# Imports

import os
import re
import sys
import json
import logging
import unicodedata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Env
from dotenv import load_dotenv

# OpenSearch
from opensearchpy import OpenSearch, RequestsHttpConnection

# ML
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

print('Imports OK')

In [ ]:
# Chargement du fichier .env

env_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '.env')
loaded = load_dotenv(dotenv_path=env_path, override=True)
if not loaded:
    # fallback: chercher dans le répertoire courant
    load_dotenv(override=True)

OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
OPENSEARCH_PORT = int(os.getenv('OPENSEARCH_PORT', 9200))
OPENSEARCH_USER = os.getenv('OPENSEARCH_USER', 'admin')
OPENSEARCH_PASSWORD = os.getenv('OPENSEARCH_PASSWORD', 'R@gTime2026#Store')
INDEX_NAME = os.getenv('INDEX_NAME', 'logistore_tickets')

print(f'Host: {OPENSEARCH_HOST}:{OPENSEARCH_PORT}')
print(f'Index: {INDEX_NAME}')
print(f'User: {OPENSEARCH_USER}')

In [ ]:
# Connexion à OpenSearch

client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl=True,
    verify_certs=False,
    ssl_show_warn=False,
    connection_class=RequestsHttpConnection,
)

try:
    info = client.info()
    print('OpenSearch connecté')
    print(f"  Version : {info['version']['number']}")
    print(f"  Cluster : {info['cluster_name']}")
except Exception as e:
    print(f'Connexion OpenSearch échouée : {e}')
    print('   Les sections sémantiques utiliseront un mode dégradé.')

In [ ]:
# Chargement du dataset brut

CSV_PATH = '../data/raw/dataset-tickets-multi-lang3-4k.csv'

try:
    df = pd.read_csv(CSV_PATH)
    print(f'Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes')
    print(f'Colonnes : {list(df.columns)}')
    display(df.head(3))
except FileNotFoundError:
    print(f'Fichier non trouvé : {CSV_PATH}')
    print('Création d\'un DataFrame vide pour les tests.')
    # Colonnes standard
    cols = ['subject', 'body', 'answer', 'type', 'queue', 'priority',
            'language', 'business_type'] + [f'tag_{i}' for i in range(1, 10)]
    df = pd.DataFrame(columns=cols)
except Exception as e:
    print(f'Erreur chargement CSV : {e}')
    df = pd.DataFrame()

## Section 1: Preprocessing léger


Le preprocessing est volontairement **léger** pour préserver le signal sémantique :

- **Lowercase**: normalisation de casse
- **Suppression boilerplate**: formules de politesse multilingues (signatures, salutations)
- **Anonymisation PII**: emails, téléphones, numéros de commande
- **Normalisation whitespace**: suppression espaces multiples, sauts de ligne excessifs

Les modèles de langue multilingues (MiniLM) sont robustes aux variations mineures; mais un preprocessing trop agressif (stemming, stopwords) dégrade les embeddings.

In [ ]:
# Patterns boilerplate multilingues

BOILERPLATE_PATTERNS = [
    # EN
    r'best\s+regards[,.]?.*',
    r'kind\s+regards[,.]?.*',
    r'sincerely[,.]?.*',
    r'thank\s+you\s+for\s+your\s+(email|message|contact)[,.]?.*',
    r'please\s+do\s+not\s+hesitate\s+to\s+contact\s+us.*',
    r'if\s+you\s+have\s+any\s+(further\s+)?(questions|concerns).*',
    # FR
    r'cordialement[,.]?.*',
    r'bien\s+cordialement[,.]?.*',
    r'avec\s+nos\s+meilleures\s+salutations[,.]?.*',
    r'n\'hésitez\s+pas\s+à\s+nous\s+contacter.*',
    r'veuillez\s+agréer.*',
    r'dans\s+l\'attente\s+de\s+votre\s+retour.*',
    # DE
    r'mit\s+freundlichen\s+gr[üu][ß]en[,.]?.*',
    r'freundliche\s+gr[üu][ß]e[,.]?.*',
    r'mit\s+besten\s+gr[üu][ß]en[,.]?.*',
    r'hochachtungsvoll[,.]?.*',
    r'zögern\s+sie\s+nicht.*',
    # ES
    r'saludos\s+cordiales[,.]?.*',
    r'atentamente[,.]?.*',
    r'un\s+cordial\s+saludo[,.]?.*',
    r'no\s+dude\s+en\s+contactarnos.*',
    # PT
    r'atenciosamente[,.]?.*',
    r'com\s+os\s+melhores\s+cumprimentos[,.]?.*',
    r'n[ãa]o\s+hesite\s+em\s+contactar.*',
    r'aguardamos\s+o\s+seu\s+retorno.*',
]

# Compile patterns (IGNORECASE + DOTALL pour multi-lignes)
_BOILERPLATE_RE = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in BOILERPLATE_PATTERNS]

# PII patterns
_EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+', re.IGNORECASE)
_PHONE_RE = re.compile(
    r'(?:\+?\d[\s.-]?)?(?:\(?\d{2,4}\)?[\s.-]?){3,5}\d{2,4}'
)
_ORDER_RE = re.compile(r'(?:ORDER[-#]?\s*|#)\d{4,10}', re.IGNORECASE)
_EXCESS_WS = re.compile(r'[ \t]{2,}')
_EXCESS_NL = re.compile(r'\n{3,}')

print('Patterns compilés')

In [ ]:
def clean_text(text: str, lang: str = 'EN') -> str:
    """Nettoie un texte ticket: lowercase, suppression boilerplate & PII, normalisation.

    Args:
        text: Texte brut du ticket.
        lang: Code langue (EN, FR, DE, ES, PT) — non utilisé ici mais conservé pour extension.

    Returns:
        Texte nettoyé.
    """
    if not isinstance(text, str):
        return ''

    # 1. Normalisation unicode (NFC) + lowercase
    text = unicodedata.normalize('NFC', text)
    text = text.lower()

    # 2. Suppression boilerplate multilingue
    for pattern in _BOILERPLATE_RE:
        text = pattern.sub('', text)

    # 3. Anonymisation PII
    text = _EMAIL_RE.sub('[EMAIL]', text)
    text = _PHONE_RE.sub('[PHONE]', text)
    text = _ORDER_RE.sub('[ORDER_ID]', text)

    # 4. Normalisation whitespace
    text = _EXCESS_NL.sub('\n\n', text)
    text = _EXCESS_WS.sub(' ', text)
    text = text.strip()

    return text


# Test rapide
sample = "Dear support team, ORDER-12345 failed. Email me at john.doe@example.com. Best regards, John"
print('Avant :', sample)
print('Après :', clean_text(sample, 'EN'))


In [ ]:
def preprocess_ticket(row) -> dict:
    """Prétraite une ligne du DataFrame tickets.

    Concatène subject + body + answer, applique clean_text,
    conserve les métadonnées catégorielles.

    Args:
        row: pd.Series — une ligne du DataFrame.

    Returns:
        dict avec text_clean et métadonnées.
    """
    lang = str(row.get('language', 'EN')).upper()

    # Concaténation subject | body | answer
    parts = [
        str(row.get('subject', '') or ''),
        str(row.get('body', '') or ''),
        str(row.get('answer', '') or ''),
    ]
    raw_text = ' | '.join(p for p in parts if p.strip())

    # Nettoyage
    text_clean = clean_text(raw_text, lang)

    # Métadonnées catégorielles
    meta = {
        'text_clean': text_clean,
        'text_raw': raw_text,
        'subject': str(row.get('subject', '') or ''),
        'body': str(row.get('body', '') or ''),
        'answer': str(row.get('answer', '') or ''),
        'language': lang,
        'type': str(row.get('type', '') or ''),
        'queue': str(row.get('queue', '') or ''),
        'priority': str(row.get('priority', '') or ''),
        'business_type': str(row.get('business_type', '') or ''),
    }
    # Tags tag_1 .. tag_9
    for i in range(1, 10):
        col = f'tag_{i}'
        meta[col] = str(row.get(col, '') or '')

    return meta


print('Fonction preprocess_ticket définie')

In [ ]:
# Application au DataFrame

if not df.empty:
    print('Preprocessing en cours...')
    df_clean = df.apply(preprocess_ticket, axis=1, result_type='expand')
    print(f'Preprocessing terminé : {len(df_clean)} tickets')
else:
    print('DataFrame vide, création d\'un df_clean vide')
    cols = ['text_clean', 'text_raw', 'subject', 'body', 'answer',
            'language', 'type', 'queue', 'priority', 'business_type']
    cols += [f'tag_{i}' for i in range(1, 10)]
    df_clean = pd.DataFrame(columns=cols)

In [ ]:
# Statistiques avant/après nettoyage

if not df_clean.empty:
    df_clean['len_raw'] = df_clean['text_raw'].str.len()
    df_clean['len_clean'] = df_clean['text_clean'].str.len()

    stats = df_clean[['len_raw', 'len_clean']].describe().loc[['mean', '50%', 'max']]
    stats.index = ['Moyenne', 'Médiane', 'Max']
    print('Longueur des textes (en caractères) :')
    display(stats.rename(columns={'len_raw': 'Avant nettoyage', 'len_clean': 'Après nettoyage'}))

    reduction = (1 - df_clean['len_clean'].sum() / df_clean['len_raw'].sum()) * 100
    print(f'\nRéduction moyenne: {reduction:.1f}%')

    # Exemple par langue
    print('\n--- Exemples avant/après par langue ---')
    for lang in ['EN', 'FR', 'DE', 'ES', 'PT']:
        subset = df_clean[df_clean['language'] == lang]
        if len(subset) > 0:
            row = subset.iloc[0]
            print(f'\n[{lang}]')
            print(f'  AVANT  : {row["text_raw"][:150]}...')
            print(f'  APRÈS  : {row["text_clean"][:150]}...')
else:
    print('df_clean vide, statistiques non disponibles')

In [ ]:
# Sauvegarde du DataFrame nettoyé

import pathlib

output_dir = pathlib.Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'tickets_preprocessed.csv'
if not df_clean.empty:
    df_clean.to_csv(output_path, index=False)
    print(f'Sauvegardé : {output_path}  ({len(df_clean)} lignes)')
else:
    print('DataFrame vide, pas de sauvegarde')

## Section 2: Text-to-text avec BM25 Search

On utilise une fonction de ranking probabiliste basée sur la fréquence des termes:

$$
\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

avec :

$$
\text{IDF}(q_i) = \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)
$$

**Légende :**

| Symbole | Signification |
|---|---|
| $Q = \{q_1, \dots, q_n\}$ | Requête décomposée en $n$ termes |
| $D$ | Document scoré |
| $f(q_i, D)$ | Fréquence du terme $q_i$ dans le document $D$ (term frequency) |
| $\text{IDF}(q_i)$ | Fréquence inverse de document — pénalise les termes trop communs |
| $N$ | Nombre total de documents dans le corpus |
| $n(q_i)$ | Nombre de documents contenant le terme $q_i$ |
| $\|D\|$ | Longueur du document $D$ (en tokens) |
| $\text{avgdl}$ | Longueur moyenne des documents dans le corpus |
| $k_1$ | Paramètre de saturation de la fréquence (typiquement $1.2$–$2.0$) |
| $b$ | Paramètre de normalisation par la longueur (typiquement $0.75$) |

**Intuition des paramètres :**
- $k_1$ contrôle à quelle vitesse l'apport d'un terme répété sature — plus $k_1$ est grand, plus répéter un mot booste le score
- $b = 1$ normalise complètement par la longueur du document, $b = 0$ ignore la longueur
- L'IDF donne un poids élevé aux termes rares et pénalise les termes très fréquents (stopwords)

- **Forces** : rapide, interprétable, excellent pour mots-clés exacts
- **Faiblesses** : pas de compréhension sémantique, sensible aux synonymes
- **Usage** : baseline lexical, complément du sémantique dans l'hybride

Ici on utilise `rank_bm25` avec filtrage par langue.

In [ ]:
# Construction de l'index BM25 par langue

def tokenize(text: str) -> list:
    """Tokenisation simple : lowercase + split sur espaces et ponctuation."""
    text = text.lower()
    tokens = re.split(r'[\s\.,;:!?\(\)\[\]\{\}"\'/\\-]+', text)
    return [t for t in tokens if len(t) > 1]


# Index BM25 global
BM25_INDEX = {}  # lang -> {'bm25': BM25Okapi, 'indices': list[int]}

if not df_clean.empty:
    for lang in df_clean['language'].unique():
        lang_mask = df_clean['language'] == lang
        lang_indices = df_clean.index[lang_mask].tolist()
        corpus = df_clean.loc[lang_mask, 'text_clean'].fillna('').tolist()
        tokenized = [tokenize(doc) for doc in corpus]
        # Eviter corpus vide
        tokenized = [t if t else ['<empty>'] for t in tokenized]
        bm25 = BM25Okapi(tokenized)
        BM25_INDEX[lang] = {
            'bm25': bm25,
            'indices': lang_indices,
            'corpus': corpus,
        }
    print(f'Index BM25 construit pour {len(BM25_INDEX)} langues : {list(BM25_INDEX.keys())}')
    for lang, idx in BM25_INDEX.items():
        print(f'   {lang}: {len(idx["indices"])} documents')
else:
    print('df_clean vide, index BM25 non construit')

In [ ]:
def bm25_search(query: str, lang: str = 'EN', top_k: int = 5) -> list:
    """Recherche BM25 sur le sous-corpus d'une langue donnée.

    Args:
        query: Requête texte.
        lang: Code langue (EN, FR, DE, ES, PT).
        top_k: Nombre de résultats à retourner.

    Returns:
        Liste de dicts {score, subject, body_preview, language, type, idx}.
    """
    lang = lang.upper()
    if lang not in BM25_INDEX:
        print(f'Langue {lang} non indexée. Langues disponibles : {list(BM25_INDEX.keys())}')
        return []

    idx_data = BM25_INDEX[lang]
    bm25 = idx_data['bm25']
    indices = idx_data['indices']

    # Tokenisation query
    tokenized_query = tokenize(query)
    if not tokenized_query:
        return []

    # Scores BM25
    scores = bm25.get_scores(tokenized_query)

    # Top-k indices
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for local_idx in top_indices:
        global_idx = indices[local_idx]
        row = df_clean.loc[global_idx]
        results.append({
            'score': float(scores[local_idx]),
            'subject': row.get('subject', ''),
            'body_preview': str(row.get('body', ''))[:200],
            'language': row.get('language', ''),
            'type': row.get('type', ''),
            'priority': row.get('priority', ''),
            'idx': global_idx,
        })

    return results


print('Fonction bm25_search définie')

In [ ]:
# Tests BM25 : 3 requêtes tests (EN, FR, DE)

bm25_queries = [
    ('My order has not arrived yet, tracking shows no update', 'EN'),
    ('Mon colis est endommagé, je veux un remboursement', 'FR'),
    ('Das Paket wurde an die falsche Adresse geliefert', 'DE'),
]

for query, lang in bm25_queries:
    print(f'\nRequête [{lang}]: "{query}"')
    results = bm25_search(query, lang=lang, top_k=5)
    if results:
        df_res = pd.DataFrame(results)[['score', 'language', 'type', 'priority', 'subject', 'body_preview']]
        display(df_res)
    else:
        print('   Aucun résultat (corpus vide ?)')

### Remarques sur BM25

- BM25 performe bien sur les termes exacts (ex: "remboursement", "Adresse").
- Les stopwords non filtrés peuvent biaiser les scores, d'où le filtrage par langue qui aide à corriger ce biais.
- Pour les requêtes paraphrasées (synonymes, formulations différentes), BM25 est limité.
- 
- **Conclusion**: excellent baseline, mais insuffisant seul pour la recherche sémantique multilingue.

## Section 3: Categorical Search (filtrage par métadonnées)


Le filtrage catégoriel permet de restreindre l'espace de recherche avant le ranking:

| Colonne | Exemples de valeurs |
|---------|--------------------|
| `type` | incident, question, problem, task |
| `queue` | support, billing, technical, logistics |
| `priority` | low, medium, high, urgent |
| `business_type` | B2B, B2C |
| `tag_1..9` | tags libres |

La recherche catégorielle combine : **filtre dur** (AND logique) + **BM25** sur le sous-corpus filtré.

In [ ]:
def categorical_filter(df_input: pd.DataFrame, filters: dict) -> pd.DataFrame:
    """Filtre un DataFrame de tickets selon des critères catégoriels (AND logique).

    Args:
        df_input: DataFrame à filtrer (df_clean ou sous-ensemble).
        filters: Dict de filtres, ex: {'language': 'EN', 'type': 'incident', 'priority': 'high'}.
                 Les valeurs sont case-insensitive.

    Returns:
        Sous-ensemble filtré du DataFrame.
    """
    result = df_input.copy()
    for col, value in filters.items():
        if col not in result.columns:
            print(f'Colonne inconnue ignorée : {col}')
            continue
        if isinstance(value, list):
            # Filtre multi-valeurs (OR au sein du filtre)
            values_lower = [str(v).lower() for v in value]
            result = result[result[col].str.lower().isin(values_lower)]
        else:
            result = result[result[col].str.lower() == str(value).lower()]
    return result


# Test du filtre
if not df_clean.empty:
    test_filter = {'language': 'EN', 'type': 'incident'}
    filtered = categorical_filter(df_clean, test_filter)
    print(f'Filtre {test_filter} → {len(filtered)} tickets')
    if not filtered.empty:
        print('Distribution priorités :')
        display(filtered['priority'].value_counts().to_frame('count'))
else:
    print('df_clean vide')

In [ ]:
def categorical_search(query: str, filters: dict, top_k: int = 5) -> list:
    """Recherche combinée : filtre catégoriel + BM25 sur le sous-corpus.

    Args:
        query: Requête texte.
        filters: Dict de filtres catégoriels.
        top_k: Nombre de résultats.

    Returns:
        Liste de dicts résultats avec score BM25 et métadonnées.
    """
    if df_clean.empty:
        return []

    # 1. Filtrage catégoriel
    filtered_df = categorical_filter(df_clean, filters)
    if filtered_df.empty:
        print(f'Aucun ticket après filtrage : {filters}')
        return []

    # 2. Index BM25 sur le sous-corpus filtré
    corpus = filtered_df['text_clean'].fillna('').tolist()
    tokenized = [tokenize(doc) for doc in corpus]
    tokenized = [t if t else ['<empty>'] for t in tokenized]
    bm25_local = BM25Okapi(tokenized)

    # 3. Score sur la requête
    tokenized_query = tokenize(query)
    if not tokenized_query:
        return []
    scores = bm25_local.get_scores(tokenized_query)

    # 4. Top-k
    top_local_indices = np.argsort(scores)[::-1][:top_k]
    global_indices = filtered_df.index.tolist()

    results = []
    for local_idx in top_local_indices:
        global_idx = global_indices[local_idx]
        row = df_clean.loc[global_idx]
        results.append({
            'score': float(scores[local_idx]),
            'subject': row.get('subject', ''),
            'body_preview': str(row.get('body', ''))[:200],
            'language': row.get('language', ''),
            'type': row.get('type', ''),
            'queue': row.get('queue', ''),
            'priority': row.get('priority', ''),
            'business_type': row.get('business_type', ''),
            'idx': global_idx,
        })

    return results


print('Fonction categorical_search définie')

In [ ]:
# Test 1: incidents haute priorité EN

print('=== Test 1 : Incidents haute priorité EN ===')
q1 = 'order not received, urgent help needed'
f1 = {'language': 'EN', 'type': 'incident', 'priority': 'high'}
res1 = categorical_search(q1, f1, top_k=5)
if res1:
    display(pd.DataFrame(res1)[['score', 'language', 'type', 'priority', 'queue', 'subject']])
    print(f'\nDistribution queues :')
    display(pd.DataFrame(res1)['queue'].value_counts().to_frame('count'))
else:
    print('Aucun résultat')

print('\n=== Test 2 : Questions DE ===')
q2 = 'Wie kann ich meine Bestellung stornieren'
f2 = {'language': 'DE', 'type': 'question'}
res2 = categorical_search(q2, f2, top_k=5)
if res2:
    display(pd.DataFrame(res2)[['score', 'language', 'type', 'priority', 'queue', 'subject']])
    print(f'\nDistribution types dans résultats :')
    display(pd.DataFrame(res2)['type'].value_counts().to_frame('count'))
else:
    print('Aucun résultat')

## Section 4: Semantic Search (vectoriel via OpenSearch)

### Recherche vectorielle k-NN

La recherche sémantique encode la requête en vecteur dense (dim=384) et cherche les voisins
les plus proches via **HNSW** (Hierarchical Navigable Small World) avec similarité cosinus.

```
query → encoder → vecteur q (384d)
            ↓
     OpenSearch k-NN (HNSW)
            ↓
     top-k vecteurs les plus proches
```

**Avantages** :
- Comprend les paraphrases et synonymes
- Multilingue nativement (MiniLM)
- Robuste aux variations orthographiques

**Inconvénients** :
- Plus lent que BM25
- Nécessite OpenSearch k-NN activé
- Moins précis sur les termes exacts rares


In [ ]:
# Chargement du modèle d'embedding

MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'

try:
    model = SentenceTransformer(MODEL_NAME)
    # Test rapide
    test_vec = model.encode(['test'])[0]
    print(f'Modèle chargé : {MODEL_NAME}')
    print(f' Dimension : {len(test_vec)}')
except Exception as e:
    print(f' Erreur chargement modèle : {e}')
    model = None

In [ ]:
def semantic_search(query: str, lang: str = None, top_k: int = 5) -> list:
    """Recherche sémantique k-NN via OpenSearch.

    Args:
        query: Requête texte.
        lang: Code langue pour filtre (None = toutes langues).
        top_k: Nombre de résultats.

    Returns:
        Liste de dicts {score, subject, body_preview, language, type, _id}.
    """
    if model is None:
        print('Modèle non chargé')
        return []

    # 1. Encodage de la requête
    query_vector = model.encode([query])[0].tolist()

    # 2. Construction de la query OpenSearch
    if lang:
        lang = lang.lower()
        os_query = {
            "size": top_k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": query_vector,
                        "k": top_k * 10,  # large pour compenser filtre HNSW
                        "filter": {"term": {"language": lang}}  # filtre natif knn
                    }
                }
            },
            "_source": {"excludes": ["embedding"]},
        }
    else:
        os_query = {
            "size": top_k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": query_vector,
                        "k": top_k
                    }
                }
            },
            "_source": {"excludes": ["embedding"]},
        }

    # 3. Exécution
    try:
        response = client.search(index=INDEX_NAME, body=os_query)
        hits = response['hits']['hits']
    except Exception as e:
        print(f'Erreur OpenSearch : {e}')
        return []

    # 4. Formatage résultats
    results = []
    for hit in hits:
        src = hit.get('_source', {})
        results.append({
            'score': float(hit.get('_score', 0.0)),
            '_id': hit.get('_id', ''),
            'subject': src.get('subject', ''),
            'body_preview': str(src.get('body', ''))[:200],
            'language': src.get('language', ''),
            'type': src.get('type', ''),
            'priority': src.get('priority', ''),
            'queue': src.get('queue', ''),
        })

    return results


print('Fonction semantic_search définie')

In [ ]:
# Diagnostic semantic_search
import traceback

query = "My shipment is missing, I cannot find it anywhere"
lang = "EN"

# 1. Test embedding
try:
    vec = model.encode([query])[0].tolist()
    print(f"Embedding OK — dim={len(vec)}, first values: {vec[:3]}")
except Exception as e:
    print(f"Embedding FAIL: {e}")

# 2. Test connexion OpenSearch
try:
    info = client.info()
    print(f"OpenSearch OK — version: {info['version']['number']}")
except Exception as e:
    print(f"OpenSearch FAIL: {e}")

# 3. Test index + doc count
try:
    count = client.count(index="logistore_tickets")
    print(f"Index OK — {count['count']} documents")
except Exception as e:
    print(f"Index FAIL: {e}")

# 4. Test requête kNN brute (sans filtre langue)
try:
    body = {
        "size": 3,
        "query": {
            "knn": {
                "embedding": {
                    "vector": vec,
                    "k": 3
                }
            }
        }
    }
    resp = client.search(index="logistore_tickets", body=body)
    hits = resp["hits"]["hits"]
    print(f"kNN sans filtre — {len(hits)} résultats")
    for h in hits:
        print(f"   score={h['_score']:.4f} | lang={h['_source'].get('language')} | subject={h['_source'].get('subject','')[:60]}")
except Exception as e:
    print(f"kNN FAIL: {e}")
    traceback.print_exc()

# 5. Test avec filtre langue
try:
    body_filtered = {
        "size": 3,
        "query": {
            "bool": {
                "must": [{"knn": {"embedding": {"vector": vec, "k": 3}}}],
                "filter": [{"term": {"language": lang}}]
            }
        }
    }
    resp2 = client.search(index="logistore_tickets", body=body_filtered)
    hits2 = resp2["hits"]["hits"]
    print(f"kNN avec filtre langue={lang} — {len(hits2)} résultats")
except Exception as e:
    print(f"kNN filtré FAIL: {e}")
    traceback.print_exc()

In [ ]:
resp = client.search(index="logistore_tickets", body={
    "size": 0,
    "aggs": {"langs": {"terms": {"field": "language", "size": 10}}}
})
for b in resp["aggregations"]["langs"]["buckets"]:
    print(b["key"], "→", b["doc_count"])

In [ ]:
# Tests sémantiques: requêtes paraphrasées/conceptuelles

semantic_queries = [
    ('My shipment is missing, I cannot find it anywhere', 'EN'),
    ('Je veux annuler et récupérer mon argent', 'FR'),
    ('Lieferung beschädigt angekommen', 'DE'),
]

for query, lang in semantic_queries:
    print(f'\nRequête sémantique [{lang}]: "{query}"')
    results = semantic_search(query, lang=lang, top_k=5)
    if results:
        display(pd.DataFrame(results)[['score', 'language', 'type', 'priority', 'subject']])
    else:
        print('   Aucun résultat (OpenSearch non disponible ?)')

In [ ]:
# Comparaison BM25 vs Semantic sur une même requête

comparison_query = 'package lost during delivery, no tracking update'
comparison_lang = 'EN'

print(f'Requête : "{comparison_query}" [{comparison_lang}]')
print('=' * 60)

bm25_res = bm25_search(comparison_query, lang=comparison_lang, top_k=5)
sem_res = semantic_search(comparison_query, lang=comparison_lang, top_k=5)

# Tableau côte à côte
max_len = max(len(bm25_res), len(sem_res))

comparison_rows = []
for i in range(max_len):
    bm25_entry = bm25_res[i] if i < len(bm25_res) else {}
    sem_entry = sem_res[i] if i < len(sem_res) else {}
    comparison_rows.append({
        'Rang': i + 1,
        'BM25 Score': round(bm25_entry.get('score', 0), 4),
        'BM25 Subject': bm25_entry.get('subject', '')[:50],
        'Semantic Score': round(sem_entry.get('score', 0), 4),
        'Semantic Subject': sem_entry.get('subject', '')[:50],
})

display(pd.DataFrame(comparison_rows))

---
## Section 5 — Hybrid Search avec RRF

### Reciprocal Rank Fusion (Cormack et al., 2009)

RRF combine plusieurs classements en calculant pour chaque document :

\[
\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + r(d)}
\]

Avec \(k = 60\) (valeur empiriquement robuste), \(r(d)\) = rang du document \(d\) dans le classement \(r\).

**Avantages de RRF** :
- Ne dépend pas des scores bruts (pas de normalisation nécessaire)
- Robuste aux différences d'échelle entre BM25 et cosine similarity
- Simple et efficace

**Référence** : Cormack, Clarke & Buettcher (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods.*

In [ ]:
# Import depuis src/ si disponible, sinon implémentation inline
import sys

_hybrid_module_loaded = False

try:
    sys.path.insert(0, '../')
    from src.retrieval.hybrid_search import (
        hybrid_search,
        bm25_search as hs_bm25,
        semantic_search as hs_semantic,
    )
    _hybrid_module_loaded = True
    print('Module hybrid_search importé depuis src/')
except ImportError as e:
    print(f'Module src non disponible ({e}), implémentation inline utilisée')

In [ ]:
# Implémentation RRF inline

def rrf(rankings: list, k: int = 60) -> dict:
    """Reciprocal Rank Fusion sur plusieurs classements.

    Args:
        rankings: Liste de listes d'identifiants de documents (str/int),
                  ordonnées par pertinence décroissante.
        k: Constante de lissage (défaut 60, recommandé par Cormack 2009).

    Returns:
        Dict {doc_id: rrf_score} trié par score décroissant.
    """
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            doc_id = str(doc_id)
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


print('Fonction RRF définie (k=60)')

# Test RRF
test_rankings = [['A', 'B', 'C'], ['B', 'A', 'D'], ['C', 'B', 'A']]
test_rrf = rrf(test_rankings)
print('Test RRF :', test_rrf)

In [ ]:
def hybrid_rrf_search(query: str, lang: str = None, top_k: int = 5,
                       candidate_k: int = 20) -> list:
    """Recherche hybride BM25 + Semantic via RRF.

    Args:
        query: Requête texte.
        lang: Code langue (None = toutes langues).
        top_k: Nombre de résultats finaux.
        candidate_k: Nombre de candidats par méthode.

    Returns:
        Liste de dicts avec score RRF, rangs individuels et métadonnées.
    """
    if _hybrid_module_loaded:
        # Utilise le module src/ si disponible
        try:
            return hybrid_search(query=query, lang=lang, top_k=top_k)
        except Exception as e:
            print(f'Erreur module hybrid_search : {e}, fallback inline')

    # Implémentation inline

    # 1. BM25 candidates
    if lang:
        lang = lang.lower()
        bm25_results = bm25_search(query, lang=lang, top_k=candidate_k)
    else:
        # Toutes langues : agréger
        bm25_results = []
        for l in BM25_INDEX.keys():
            bm25_results.extend(bm25_search(query, lang=l, top_k=candidate_k // 5))

    # 2. Semantic candidates
    sem_results = semantic_search(query, lang=lang, top_k=candidate_k)

    # 3. Construction des classements
    bm25_ids = [str(r['idx']) for r in bm25_results]
    sem_ids = [r['_id'] for r in sem_results]

    # Mapping id → données
    doc_map = {}
    for r in bm25_results:
        doc_map[str(r['idx'])] = {'source': 'bm25', **r}
    for r in sem_results:
        if r['_id'] not in doc_map:
            doc_map[r['_id']] = {'source': 'semantic', **r}

    # 4. RRF fusion
    rrf_scores = rrf([bm25_ids, sem_ids], k=60)

    # 5. Enrichissement avec rangs
    results = []
    bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ids)}
    sem_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sem_ids)}

    for doc_id, rrf_score in list(rrf_scores.items())[:top_k]:
        doc_data = doc_map.get(doc_id, {})
        results.append({
            'rrf_score': round(rrf_score, 6),
            'bm25_rank': bm25_rank_map.get(doc_id, 'N/A'),
            'semantic_rank': sem_rank_map.get(doc_id, 'N/A'),
            'subject': doc_data.get('subject', ''),
            'language': doc_data.get('language', ''),
            'type': doc_data.get('type', ''),
            'priority': doc_data.get('priority', ''),
            '_id': doc_id,
        })

    return results


print('Fonction hybrid_rrf_search définie')

In [ ]:
# Tests: comparaison BM25, Semantic et Hybrid sur 3 requêtes

hybrid_test_queries = [
    ('My printer is only printing blank pages. What should I do?', 'EN'),
    ("mon Mac Book Air fait un bruit fort à l'allumage, que faire?", 'FR'),
    ('Mein DELL-Computer lässt sich nicht mehr einschalten und bleibt auch nach langem Drücken des Netzschalters ausgeschaltet. Was kann ich tun?', 'DE'),
]

for query, lang in hybrid_test_queries:
    print(f'\nHybrid Search [{lang}]: "{query}"')
    hybrid_res = hybrid_rrf_search(query, lang=lang, top_k=5)

    if hybrid_res:
        df_hybrid = pd.DataFrame(hybrid_res)[[
            'rrf_score', 'bm25_rank', 'semantic_rank', 'language', 'type', 'subject'
        ]]
        display(df_hybrid)
    else:
        print('   Aucun résultat')

In [ ]:
# Tableau comparatif BM25, Semantic, Hybrid (rang)

comp_query = 'My printer is only printing blank pages. What should I do?'
comp_lang = 'EN'

print(f'Comparaison complète pour : "{comp_query}" [{comp_lang}]')

bm25_res_comp = bm25_search(comp_query, lang=comp_lang, top_k=10)
sem_res_comp = semantic_search(comp_query, lang=comp_lang, top_k=10)
hybrid_res_comp = hybrid_rrf_search(comp_query, lang=comp_lang, top_k=10)

# Réunir tous les sujets mentionnés
all_subjects = {}
for r in bm25_res_comp:
    all_subjects[str(r['idx'])] = r.get('subject', '')[:60]
for r in sem_res_comp:
    all_subjects[r['_id']] = r.get('subject', '')[:60]
for r in hybrid_res_comp:
    all_subjects[r['_id']] = r.get('subject', '')[:60]

bm25_rank_map_c = {str(r['idx']): i+1 for i, r in enumerate(bm25_res_comp)}
sem_rank_map_c = {r['_id']: i+1 for i, r in enumerate(sem_res_comp)}
hybrid_rank_map_c = {r['_id']: i+1 for i, r in enumerate(hybrid_res_comp)}

rows = []
seen = set()
# Priorité aux résultats hybrid
for doc_id in list(hybrid_rank_map_c.keys())[:10]:
    if doc_id not in seen:
        seen.add(doc_id)
        rows.append({
            'Subject': all_subjects.get(doc_id, doc_id)[:60],
            'BM25 Rank': bm25_rank_map_c.get(doc_id, '-'),
            'Semantic Rank': sem_rank_map_c.get(doc_id, '-'),
            'Hybrid Rank': hybrid_rank_map_c.get(doc_id, '-'),
        })

display(pd.DataFrame(rows))

---
## Section 6 — Multimodal Ranking (vectoriel + catégoriel)

### Principe

Le ranking multimodal combine :
- **Score sémantique** (0–1, cosine similarity normalisée) : pertinence de contenu
- **Boost catégoriel** (0–1) : adéquation aux filtres métadonnées

**Score final** :
\[
\text{score\_final}(d) = \alpha \cdot \text{score\_sémantique}(d) + (1-\alpha) \cdot \text{boost\_catégoriel}(d)
\]

Avec \(\alpha = 0.7\) par défaut (poids sémantique dominant).

| Boost catégoriel | Condition |
|-----------------|----------|
| 1.0 | Tous les filtres matchent |
| 0.5 | Match partiel (≥50% des filtres) |
| 0.0 | Aucun filtre ne matche |

**Cas d'usage** : requêtes avec contraintes métier fortes (ex: B2B + high priority + incident).


In [ ]:
def compute_categorical_boost(row: pd.Series, filters: dict) -> float:
    """Calcule un boost catégoriel pour un document selon des filtres.

    Args:
        row: Ligne du DataFrame (pd.Series).
        filters: Dict de filtres catégoriels.

    Returns:
        Float entre 0 et 1 (0.0=aucun match, 0.5=partiel, 1.0=total).
    """
    if not filters:
        return 1.0

    matches = 0
    total = len(filters)

    for col, value in filters.items():
        if col not in row.index:
            total -= 1  # Colonne inconnue : ignore
            continue
        if isinstance(value, list):
            if str(row[col]).lower() in [str(v).lower() for v in value]:
                matches += 1
        else:
            if str(row[col]).lower() == str(value).lower():
                matches += 1

    if total == 0:
        return 1.0

    ratio = matches / total
    if ratio == 1.0:
        return 1.0
    elif ratio >= 0.5:
        return 0.5
    else:
        return 0.0


def multimodal_ranking(query: str, filters: dict, top_k: int = 10,
                        alpha: float = 0.7, n_candidates: int = 50) -> list:
    """Ranking multimodal : score sémantique + boost catégoriel.

    Args:
        query: Requête texte.
        filters: Dict de filtres catégoriels (appliqués comme boost, pas filtre dur).
        top_k: Nombre de résultats finaux.
        alpha: Poids du score sémantique (0–1). Défaut 0.7.
        n_candidates: Nombre de candidats sémantiques initiaux.

    Returns:
        Liste de dicts avec score_final, semantic_score, categorical_boost.
    """
    # 1. Récupération de N candidats via semantic search (sans filtre strict)
    candidates = semantic_search(query, lang=None, top_k=n_candidates)

    if not candidates and not df_clean.empty:
        # Fallback sur BM25 si OpenSearch non disponible
        lang_filter = filters.get('language', None)
        if lang_filter:
            candidates_bm25 = bm25_search(query, lang=lang_filter, top_k=n_candidates)
        else:
            candidates_bm25 = []
            for l in list(BM25_INDEX.keys())[:5]:
                candidates_bm25.extend(bm25_search(query, lang=l, top_k=n_candidates // 5))
        # Adapter le format
        candidates = []
        for r in candidates_bm25:
            max_score = max([c['score'] for c in candidates_bm25] + [1])
            candidates.append({
                'score': r['score'] / max_score if max_score > 0 else 0,
                '_id': str(r.get('idx', '')),
                'subject': r.get('subject', ''),
                'body_preview': r.get('body_preview', ''),
                'language': r.get('language', ''),
                'type': r.get('type', ''),
                'priority': r.get('priority', ''),
                'queue': r.get('queue', ''),
            })

    if not candidates:
        return []

    # 2. Score final pour chaque candidat
    results = []
    for cand in candidates:
        sem_score = float(cand.get('score', 0.0))

        # Boost catégoriel : chercher la ligne dans df_clean
        doc_id = cand.get('_id', '')
        cat_boost = 0.5  # défaut neutre si pas trouvé

        if not df_clean.empty and filters:
            # Chercher par _id ou par subject match
            try:
                idx = int(doc_id)
                if idx in df_clean.index:
                    cat_boost = compute_categorical_boost(df_clean.loc[idx], filters)
            except (ValueError, KeyError):
                # Cherche par subject
                subj = cand.get('subject', '')
                match = df_clean[df_clean['subject'] == subj]
                if not match.empty:
                    cat_boost = compute_categorical_boost(match.iloc[0], filters)
        elif not filters:
            cat_boost = 1.0

        score_final = alpha * sem_score + (1 - alpha) * cat_boost

        results.append({
            'score_final': round(score_final, 6),
            'semantic_score': round(sem_score, 6),
            'categorical_boost': cat_boost,
            'subject': cand.get('subject', ''),
            'language': cand.get('language', ''),
            'type': cand.get('type', ''),
            'priority': cand.get('priority', ''),
            '_id': doc_id,
        })

    # 3. Tri par score final
    results.sort(key=lambda x: x['score_final'], reverse=True)
    return results[:top_k]


print('Fonctions multimodal_ranking et compute_categorical_boost définies')

In [ ]:
# Tests multimodal ranking
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

mm_queries = [
    ('urgent delivery problem, item not received', {'language': 'EN', 'priority': 'high', 'type': 'incident'}),
    ('billing issue wrong amount charged', {'language': 'EN', 'type': 'problem', 'queue': 'billing'}),
]

for query, filters in mm_queries:
    print(f'\nMultimodal [{query[:40]}...] | Filtres: {filters}')
    mm_res = multimodal_ranking(query, filters=filters, top_k=10, alpha=0.7)

    if mm_res:
        display(pd.DataFrame(mm_res)[[
            'score_final', 'semantic_score', 'categorical_boost',
            'language', 'type', 'priority', 'subject'
        ]])
    else:
        print('   Aucun résultat')

In [ ]:
# Visualisation: scores sémantiques vs scores finaux

viz_query = 'My printer is only printing blank pages. What should I do?'
viz_filters = {'language': 'EN', 'priority': 'high', 'type': 'incident'}

viz_res = multimodal_ranking(viz_query, filters=viz_filters, top_k=8, alpha=0.7)

if viz_res:
    fig, ax = plt.subplots(figsize=(10, 5))

    x = range(len(viz_res))
    labels = [r['subject'][:30] + '...' if len(r['subject']) > 30 else r['subject']
              for r in viz_res]
    sem_scores = [r['semantic_score'] for r in viz_res]
    final_scores = [r['score_final'] for r in viz_res]

    width = 0.35
    bars1 = ax.bar([i - width/2 for i in x], sem_scores, width,
                   label='Score sémantique', color='steelblue', alpha=0.8)
    bars2 = ax.bar([i + width/2 for i in x], final_scores, width,
                   label='Score final (multimodal)', color='darkorange', alpha=0.8)

    ax.set_xlabel('Document')
    ax.set_ylabel('Score')
    ax.set_title('Multimodal Ranking : Score sémantique vs Score final\n'
                 f'Requête: "{viz_query[:50]}"')
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('../data/processed/multimodal_scores.png', dpi=100)
    plt.show()
    print('Graphique sauvegardé : ../data/processed/multimodal_scores.png')
else:
    print('Pas de résultats pour la visualisation')

---
## Section 7 — Cross-Encoder Reranking

### Bi-Encoder vs Cross-Encoder

| | Bi-Encoder | Cross-Encoder |
|--|------------|---------------|
| Principe | Encode requête et doc séparément | Encode paire (requête, doc) conjointement |
| Vitesse | Rapide (O(n)) | Lent (O(n × query)) |
| Qualité | Bonne (retrieval) | Meilleure (reranking) |
| Usage | Retrieval initial (top-K) | Reranking final (top-5) |

**Pipeline recommandé** :
```
Hybrid Search (top-20) → Cross-Encoder Reranker → top-5 final
```

**Modèle** : `cross-encoder/ms-marco-MiniLM-L-6-v2` (EN)

> Pour le multilingue, utiliser `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`
  qui est entraîné sur mMARCO (13 langues).


In [ ]:
# Chargement du cross-encoder
from sentence_transformers import CrossEncoder

CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
# Alternatif multilingue: 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

try:
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    # Test
    test_score = cross_encoder.predict([('test query', 'test document')])
    print(f'Cross-encoder chargé : {CROSS_ENCODER_MODEL}')
    print(f'   Score test : {test_score[0]:.4f}')
except Exception as e:
    print(f'Erreur chargement cross-encoder : {e}')
    cross_encoder = None

In [ ]:
def rerank_with_cross_encoder(query: str, candidates: list,
                               top_k: int = 5) -> list:
    """Reranke des candidats avec un cross-encoder.

    Args:
        query: Requête texte originale.
        candidates: Liste de dicts candidats (doivent avoir 'subject' et/ou 'body_preview').
        top_k: Nombre de résultats après reranking.

    Returns:
        Liste reranquée avec cross_encoder_score et original_rank.
    """
    if cross_encoder is None:
        print('Cross-encoder non disponible, retour des candidats originaux')
        return candidates[:top_k]

    if not candidates:
        return []

    # Construction des paires (query, doc_text)
    pairs = []
    for cand in candidates:
        doc_text = cand.get('subject', '') + ' ' + cand.get('body_preview', '')
        doc_text = doc_text.strip()[:512]  # Limite cross-encoder
        pairs.append((query, doc_text))

    # Prédiction des scores
    try:
        ce_scores = cross_encoder.predict(pairs)
    except Exception as e:
        print(f'Erreur cross-encoder predict : {e}')
        return candidates[:top_k]

    # Ajout rank original + score CE
    enriched = []
    for i, (cand, score) in enumerate(zip(candidates, ce_scores)):
        enriched.append({
            **cand,
            'original_rank': i + 1,
            'cross_encoder_score': float(score),
        })

    # Tri par score CE décroissant
    enriched.sort(key=lambda x: x['cross_encoder_score'], reverse=True)

    # Ajout rang après reranking
    for i, item in enumerate(enriched):
        item['reranked_rank'] = i + 1

    return enriched[:top_k]


print('Fonction rerank_with_cross_encoder définie')

In [ ]:
# Pipeline complet: Hybrid top-20 vers Cross-Encoder top-5

rerank_queries = [
    ('product arrived broken and the customer service ignored my complaint', 'EN'),
    ('I was charged twice for the same order', 'EN'),
]

for query, lang in rerank_queries:
    print(f'\nPipeline Reranking [{lang}]: "{query}"')
    print('-' * 60)

    # 1. Hybrid search : top 20 candidats
    candidates_20 = hybrid_rrf_search(query, lang=lang, top_k=20)

    if not candidates_20:
        # Fallback BM25
        candidates_20 = bm25_search(query, lang=lang, top_k=20)
        for c in candidates_20:
            c['_id'] = str(c.get('idx', ''))

    print(f'   Candidats hybrid : {len(candidates_20)}')

    # 2. Cross-encoder reranking : top 5
    reranked = rerank_with_cross_encoder(query, candidates_20, top_k=5)

    # 3. Affichage avant/après
    if reranked:
        print('\n   Avant/Après reranking :')
        comparison = pd.DataFrame(reranked)[[
            'original_rank', 'reranked_rank', 'cross_encoder_score',
            'language', 'type', 'subject'
        ]]
        comparison['rank_change'] = comparison['original_rank'] - comparison['reranked_rank']
        display(comparison)
    else:
        print('   Aucun résultat après reranking')

---
## Section 8 — Meta-Ranking (Rankings of Rankings)

### Principe du Meta-Ranking

Le meta-ranking est un **ensemble de rankings hétérogènes** qui combine 4 systèmes :

1. **BM25** : signal lexical
2. **Semantic k-NN** : signal sémantique dense
3. **Hybrid RRF** : fusion déjà optimisée
4. **Multimodal** : signal catégoriel + sémantique (optionnel)

**Différence avec RRF simple** : le meta-ranking peut être **pondéré** (certains systèmes
ont plus de poids) et inclut la **traçabilité de provenance** (quel système a ranké le document).

**Application** : requêtes complexes où aucun système seul n'est suffisant.

\[
\text{meta-score}(d) = \sum_{s \in S} w_s \cdot \frac{1}{k + r_s(d)}
\]

Avec \(w_s\) = poids du système \(s\) (défaut 1.0 pour RRF standard).


In [ ]:
def weighted_rrf(rankings_with_weights: list, k: int = 60) -> dict:
    """RRF pondéré : chaque ranking peut avoir un poids différent.

    Args:
        rankings_with_weights: Liste de tuples (ranking, weight, name)
            où ranking est une liste d'IDs ordonnée.
        k: Constante RRF (défaut 60).

    Returns:
        Dict {doc_id: weighted_rrf_score} trié décroissant.
    """
    scores = {}
    for ranking, weight, name in rankings_with_weights:
        for rank, doc_id in enumerate(ranking):
            doc_id = str(doc_id)
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (k + rank + 1)
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


print('Fonction weighted_rrf définie')

In [ ]:
def meta_ranking(query: str, lang: str = 'EN', filters: dict = None,
                 top_k: int = 5, candidate_k: int = 20,
                 use_cross_encoder: bool = True) -> list:
    """Meta-ranking : ensemble de 4 systèmes de recherche fusionnés par RRF pondéré.

    Args:
        query: Requête texte.
        lang: Code langue.
        filters: Filtres catégoriels optionnels (active Multimodal si fourni).
        top_k: Nombre de résultats finaux.
        candidate_k: Candidats par système.
        use_cross_encoder: Si True, reranking final par cross-encoder.

    Returns:
        Liste de dicts avec meta_score, provenance, et métadonnées.
    """
    all_results = {}   # doc_id -> dict données
    system_rankings = {}  # system_name -> [doc_id, ...]

    # Système 1: BM25
    try:
        bm25_res = bm25_search(query, lang=lang, top_k=candidate_k)
        bm25_ids = [str(r['idx']) for r in bm25_res]
        system_rankings['BM25'] = bm25_ids
        for r in bm25_res:
            doc_id = str(r['idx'])
            if doc_id not in all_results:
                all_results[doc_id] = {
                    'subject': r.get('subject', ''),
                    'language': r.get('language', ''),
                    'type': r.get('type', ''),
                    'priority': r.get('priority', ''),
                    'body_preview': r.get('body_preview', ''),
                }
    except Exception as e:
        print(f'BM25 échoué : {e}')
        system_rankings['BM25'] = []

    # Système 2: Semantic k-NN
    try:
        sem_res = semantic_search(query, lang=lang, top_k=candidate_k)
        sem_ids = [r['_id'] for r in sem_res]
        system_rankings['Semantic'] = sem_ids
        for r in sem_res:
            doc_id = r['_id']
            if doc_id not in all_results:
                all_results[doc_id] = {
                    'subject': r.get('subject', ''),
                    'language': r.get('language', ''),
                    'type': r.get('type', ''),
                    'priority': r.get('priority', ''),
                    'body_preview': r.get('body_preview', ''),
                }
    except Exception as e:
        print(f'Semantic search échoué : {e}')
        system_rankings['Semantic'] = []

    # Système 3: Hybrid RRF
    try:
        hybrid_res = hybrid_rrf_search(query, lang=lang, top_k=candidate_k)
        hybrid_ids = [r['_id'] for r in hybrid_res]
        system_rankings['Hybrid'] = hybrid_ids
        for r in hybrid_res:
            doc_id = r['_id']
            if doc_id not in all_results:
                all_results[doc_id] = {
                    'subject': r.get('subject', ''),
                    'language': r.get('language', ''),
                    'type': r.get('type', ''),
                    'priority': r.get('priority', ''),
                    'body_preview': r.get('body_preview', ''),
                }
    except Exception as e:
        print(f'Hybrid search échoué : {e}')
        system_rankings['Hybrid'] = []

    # Système 4: Multimodal (si on a les filtres)
    if filters:
        try:
            mm_res = multimodal_ranking(query, filters=filters, top_k=candidate_k, alpha=0.7)
            mm_ids = [r['_id'] for r in mm_res]
            system_rankings['Multimodal'] = mm_ids
            for r in mm_res:
                doc_id = r['_id']
                if doc_id not in all_results:
                    all_results[doc_id] = {
                        'subject': r.get('subject', ''),
                        'language': r.get('language', ''),
                        'type': r.get('type', ''),
                        'priority': r.get('priority', ''),
                        'body_preview': r.get('body_preview', ''),
                    }
        except Exception as e:
            print(f'Multimodal échoué : {e}')
            system_rankings['Multimodal'] = []
    else:
        system_rankings['Multimodal'] = []  # N/A

    # RRF pondéré
    # Poids: BM25=1.0, Semantic=1.2, Hybrid=1.5, Multimodal=1.0
    rankings_with_weights = [
        (system_rankings.get('BM25', []), 1.0, 'BM25'),
        (system_rankings.get('Semantic', []), 1.2, 'Semantic'),
        (system_rankings.get('Hybrid', []), 1.5, 'Hybrid'),
        (system_rankings.get('Multimodal', []), 1.0, 'Multimodal'),
    ]
    meta_scores = weighted_rrf(rankings_with_weights, k=60)

    # Préparation top candidats pour reranking
    top_candidate_ids = list(meta_scores.keys())[:max(top_k * 4, 20)]
    candidates_for_rerank = []
    for doc_id in top_candidate_ids:
        doc_data = all_results.get(doc_id, {})
        provenance = explain_ranking_inline(doc_id, system_rankings)
        candidates_for_rerank.append({
            'meta_score': round(meta_scores[doc_id], 8),
            '_id': doc_id,
            'subject': doc_data.get('subject', ''),
            'body_preview': doc_data.get('body_preview', ''),
            'language': doc_data.get('language', ''),
            'type': doc_data.get('type', ''),
            'priority': doc_data.get('priority', ''),
            'provenance': provenance,
        })

    # Cross-encoder reranking optionnel
    if use_cross_encoder and cross_encoder is not None and candidates_for_rerank:
        reranked = rerank_with_cross_encoder(query, candidates_for_rerank, top_k=top_k)
        return reranked
    else:
        return candidates_for_rerank[:top_k]


def explain_ranking_inline(doc_id: str, system_rankings: dict) -> str:
    """Génère une explication de provenance pour un document.

    Args:
        doc_id: Identifiant du document.
        system_rankings: Dict {system_name: [doc_ids]}.

    Returns:
        String type 'BM25: #3, Semantic: #1, Hybrid: #2, Multimodal: N/A'.
    """
    parts = []
    for system_name, ranking in system_rankings.items():
        try:
            rank = ranking.index(str(doc_id)) + 1
            parts.append(f'{system_name}: #{rank}')
        except ValueError:
            parts.append(f'{system_name}: N/A')
    return ', '.join(parts)


def explain_ranking(result: dict) -> str:
    """Retourne la provenance d'un résultat meta-ranking.

    Args:
        result: Dict résultat avec clé 'provenance'.

    Returns:
        String de provenance.
    """
    return result.get('provenance', 'Provenance non disponible')


print('Fonctions meta_ranking et explain_ranking définies')


In [ ]:
# Tests meta-ranking

meta_query = 'damaged product and no response from customer service despite multiple emails'
meta_lang = 'EN'
meta_filters = {'type': 'incident', 'priority': 'high'}

print(f'Meta-Ranking : "{meta_query}" [{meta_lang}]')
print(f'   Filtres : {meta_filters}')
print('=' * 70)

meta_res = meta_ranking(
    query=meta_query,
    lang=meta_lang,
    filters=meta_filters,
    top_k=5,
    use_cross_encoder=True,
)

if meta_res:
    # Affichage avec explications
    print('\nRésultats finaux :')
    for i, res in enumerate(meta_res, 1):
        print(f'\n#{i} [{res.get("language","?")}] [{res.get("type","?")}] [{res.get("priority","?")}]')
        print(f'   Sujet : {res.get("subject", "")[:80]}')
        print(f'   Meta-score : {res.get("meta_score", res.get("cross_encoder_score", 0))}')
        print(f'   Provenance : {explain_ranking(res)}')
else:
    print('Aucun résultat')


In [ ]:
# Visualisation: heatmap rang/système pour top 10
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print('seaborn non disponible, utilisation matplotlib fallback')

# Re-run meta_ranking sans cross-encoder pour avoir plus de provenance
meta_res_full = meta_ranking(
    query=meta_query,
    lang=meta_lang,
    filters=meta_filters,
    top_k=10,
    use_cross_encoder=False,  # Pas de reranking pour avoir les 10
)

if meta_res_full:
    # Construire la matrice rang/système
    systems = ['BM25', 'Semantic', 'Hybrid', 'Multimodal']
    labels_hm = [r['subject'][:25] + '...' if len(r.get('subject','')) > 25
                 else r.get('subject', f'Doc {i}') for i, r in enumerate(meta_res_full[:10])]

    # Extraire rangs depuis provenance
    import re as _re
    heat_data = []
    for res in meta_res_full[:10]:
        prov = res.get('provenance', '')
        row_vals = []
        for sys in systems:
            m = _re.search(rf'{sys}:\s*#?(\d+|N/A)', prov)
            if m and m.group(1) != 'N/A':
                row_vals.append(int(m.group(1)))
            else:
                row_vals.append(0)  # 0 = absent
        heat_data.append(row_vals)

    hm_array = np.array(heat_data, dtype=float)
    hm_array[hm_array == 0] = np.nan  # absent = NaN

    fig, ax = plt.subplots(figsize=(8, max(5, len(labels_hm) * 0.5)))

    if HAS_SEABORN:
        sns.heatmap(
            hm_array,
            annot=True, fmt='.0f',
            xticklabels=systems,
            yticklabels=labels_hm,
            cmap='RdYlGn_r',
            linewidths=0.5,
            cbar_kws={'label': 'Rang (1=meilleur, NaN=absent)'},
            ax=ax,
        )
    else:
        im = ax.imshow(np.nan_to_num(hm_array, nan=25), aspect='auto', cmap='RdYlGn_r')
        ax.set_xticks(range(len(systems)))
        ax.set_xticklabels(systems)
        ax.set_yticks(range(len(labels_hm)))
        ax.set_yticklabels(labels_hm, fontsize=8)
        for i in range(len(labels_hm)):
            for j in range(len(systems)):
                val = hm_array[i, j]
                text = str(int(val)) if not np.isnan(val) else '-'
                ax.text(j, i, text, ha='center', va='center', fontsize=9)
        plt.colorbar(im, ax=ax, label='Rang')

    ax.set_title(f'Meta-Ranking : Rangs par système (top 10)\nRequête: "{meta_query[:50]}"')
    plt.tight_layout()
    plt.savefig('../data/processed/meta_ranking_heatmap.png', dpi=100)
    plt.show()
    print('Heatmap sauvegardée : ../data/processed/meta_ranking_heatmap.png')
else:
    print('Pas de résultats pour la heatmap')

---
## Section 9 — Bilan & Recommandations

### Tableau récapitulatif des approches

| Méthode | Forces | Faiblesses | Cas d'usage idéal |
|---------|--------|------------|------------------|
| **BM25** | Rapide, interprétable, excellent sur mots-clés exacts | Pas de compréhension sémantique, sensible aux synonymes | Recherche par référence exacte, numéros de commande |
| **Categorical Search** | Précision métier, filtrage dur sur métadonnées | Dépend de la qualité des métadonnées, pas de ranking sémantique | Filtrage par queue/priorité/type avant recherche |
| **Semantic k-NN** | Comprend paraphrases et synonymes, multilingue natif | Plus lent, nécessite OpenSearch k-NN, moins précis sur termes rares | Requêtes conceptuelles, recherche cross-lingue |
| **Hybrid RRF** | Combine forces BM25 + sémantique, robuste, simple | Pas de prise en compte des métadonnées catégorielles | **Cas général — recommandé pour le MVP** |
| **Multimodal** | Intègre contraintes métier comme boost doux | Plus complexe à paramétrer (alpha), dépend de la qualité des métadonnées | Requêtes B2B avec contraintes métier strictes |
| **Cross-Encoder** | Meilleure précision de reranking, comprend la relation query-doc | Lent (O(n)), limitée en batch, modèle EN principalement | Reranking final sur top-20 avant présentation |
| **Meta-Ranking** | Ensemble robuste, traçabilité de provenance | Coût computationnel élevé, complexité | Cas d'usage haute valeur, requêtes ambiguës complexes |

---

### Recommandation finale pour le MVP

**Pipeline recommandé : Hybrid RRF + Cross-Encoder**

```
Requête utilisateur
       ↓
┌──────────────┐    ┌───────────────┐
│  BM25 (top20)│    │ Semantic (top20│
└──────┬───────┘    └───────┬───────┘
       └──────────┬──────────┘
              RRF Fusion
                  ↓
         Hybrid top-20
                  ↓
      Cross-Encoder Reranking
                  ↓
             Top-5 final
```

**Justification** :
- RRF est robuste sans normalisation de score
- MiniLM multilingue couvre EN, DE, FR, ES, PT sans fine-tuning
- Le cross-encoder améliore significativement la précision à faible coût (top-20 seulement)
- Pipeline testable et déboguable facilement

---

### Prochaines étapes

1. **Intégration Streamlit** : interface de recherche avec filtres catégoriels
2. **Évaluation quantitative** :
   - Créer un jeu de test annoté (query → doc relevants)
   - Calculer nDCG@5, MRR@10, Recall@20
   - Comparer les méthodes sur ce benchmark
3. **Fine-tuning** :
   - Fine-tuner MiniLM sur les paires question/answer du dataset
   - Améliorer le cross-encoder avec des paires positives/négatives
4. **Optimisation** :
   - Cache des embeddings fréquents
   - Pré-filtrage catégoriel avant semantic search
   - Réglage des hyperparamètres (k RRF, alpha multimodal, top_k cross-encoder)
5. **Monitoring** :
   - Logging des requêtes et scores
   - Détection de drift (nouvelles langues, nouveaux types)


In [ ]:
# --- Résumé des performances relatives (qualitatif) ---

summary_data = {
    'Méthode': ['BM25', 'Categorical', 'Semantic k-NN', 'Hybrid RRF',
                'Multimodal', 'Cross-Encoder', 'Meta-Ranking'],
    'Vitesse (ms)': ['<10', '<5', '50-200', '60-210', '70-220', '100-500', '200-700'],
    'Précision (est.)': ['Faible-Moyen', 'Faible', 'Moyen-Bon', 'Bon',
                         'Bon', 'Très Bon', 'Très Bon'],
    'Complexité': ['Faible', 'Très Faible', 'Moyenne', 'Moyenne',
                   'Élevée', 'Élevée', 'Très Élevée'],
    'Multilingue': ['Partiel', 'Oui', 'Oui', 'Oui', 'Oui', 'Partiel (EN)', 'Oui'],
    'Recommandation MVP': ['Composant', 'Pre-filter', 'Composant', '✅ Core',
                            'Optionnel', '✅ Reranker', 'Optionnel'],
}

df_summary = pd.DataFrame(summary_data)
display(df_summary)

print('\nPipeline recommandé pour le MVP :')
print('  BM25 (top-20) + Semantic k-NN (top-20) → RRF → Cross-Encoder (top-5)')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.retrieval.hybrid_search import hybrid_search

results = hybrid_search(query="Mon ordinateur DELL ne s'allume plus et reste éteint malgré un appui long sur le bouton d'allumage, que faire ?", top_k=3, filters={"language": "fr"})
print(results[0].keys())
print(results[0].get("_source", {}).keys())

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from src.llm.llm_client import rag_answer

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

from importlib import reload
import src.llm.llm_client as llm_mod
reload(llm_mod)
from src.llm.llm_client import call_llm, rag_answer

In [ ]:
# Normalisation des résultats hybrid_search pour rag_answer

def to_rag_tickets(hybrid_results: list[dict]) -> list[dict]:
    """Convertit les résultats hybrid_search au format attendu par rag_answer."""
    tickets = []
    for r in hybrid_results:
        src = r.get("_source", {})
        text = src.get("text", "")
        # text = "subject | body | answer" — on découpe si possible
        parts = text.split(" | ", 2)
        tickets.append({
            "subject":      parts[0].strip() if len(parts) > 0 else "",
            "body_preview": parts[1].strip()[:300] if len(parts) > 1 else text[:300],
            "answer":       parts[2].strip()[:400] if len(parts) > 2 else "",
            "language":     src.get("language", ""),
            "type":         src.get("type", ""),
            "priority":     src.get("priority", ""),
        })
    return tickets

# Pipeline RAG complet
query = "Je veux être meilleur à FC 26 sur PS5, que faire ?"
lang = "fr"

retrieval = hybrid_search(query=query, top_k=20, filters={"language": lang})
tickets = to_rag_tickets(retrieval[:5])

result = rag_answer(
    query=query,
    tickets=tickets,
    lang=lang,
    model="nvidia/nemotron-3-super-120b-a12b:free"
)

print("Réponse RAG")
print(result["answer"])
print(f"\nBasée sur {result['n_tickets_used']} tickets | modèle : {result['model']}")

In [ ]:
# Normalisation des résultats hybrid_search pour rag_answer

def to_rag_tickets(hybrid_results: list[dict]) -> list[dict]:
    """Convertit les résultats hybrid_search au format attendu par rag_answer."""
    tickets = []
    for r in hybrid_results:
        src = r.get("_source", {})
        text = src.get("text", "")
        # text = "subject | body | answer" — on découpe si possible
        parts = text.split(" | ", 2)
        tickets.append({
            "subject":      parts[0].strip() if len(parts) > 0 else "",
            "body_preview": parts[1].strip()[:300] if len(parts) > 1 else text[:300],
            "answer":       parts[2].strip()[:400] if len(parts) > 2 else "",
            "language":     src.get("language", ""),
            "type":         src.get("type", ""),
            "priority":     src.get("priority", ""),
        })
    return tickets

# Pipeline RAG complet
query = "Mon imprimante DELL ne répond plus, que faire ?"
lang = "fr"

retrieval = hybrid_search(query=query, top_k=20, filters={"language": lang})
tickets = to_rag_tickets(retrieval[:5])

result = rag_answer(
    query=query,
    tickets=tickets,
    lang=lang,
    model="nvidia/nemotron-3-super-120b-a12b:free"
)

print("Réponse RAG")
print(result["answer"])
print(f"\nBasée sur {result['n_tickets_used']} tickets | modèle : {result['model']}")

In [ ]:
# Récupération des tickets via le pipeline hybride

query = "Mon ordinateur MacBook Air ne démarre plus correctement, j'ai un écran tout noir dès l'allumage, que faire ?"
lang = "fr"

top5_tickets = hybrid_rrf_search(query, lang=lang, top_k=5)

# Appel RAG
result = rag_answer(
    query=query,
    tickets=top5_tickets,
    lang=lang
)

print(result["answer"])